# VectorBleed — Experiment Analysis

Cross-Tenant Data Leakage Research Results

In [ ]:
import json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path

sns.set_theme(style='whitegrid')
results_dir = Path('../results')

## Load Experiment Results

In [ ]:
experiments = {}
exp_dir = results_dir / 'experiments'

for f in exp_dir.glob('*.json'):
    with open(f) as fp:
        experiments[f.stem] = json.load(fp)

print(f'Loaded {len(experiments)} experiment results:')
for name, data in experiments.items():
    print(f'  {name}: {data["total_probes"]} probes, {data["detection_rate"]:.2%} detection rate')

## Embedding Space Visualization

In [ ]:
from sklearn.decomposition import PCA

emb_dir = results_dir / 'corpus' / 'embeddings'
tenant_a = np.load(emb_dir / 'tenant_a_embeddings.npy')
tenant_b = np.load(emb_dir / 'tenant_b_embeddings.npy')

all_emb = np.vstack([tenant_a, tenant_b])
pca = PCA(n_components=2)
coords = pca.fit_transform(all_emb)

fig, ax = plt.subplots(figsize=(10, 7))
ax.scatter(coords[:len(tenant_a), 0], coords[:len(tenant_a), 1], c='blue', alpha=0.6, label='Tenant A (Financial)')
ax.scatter(coords[len(tenant_a):, 0], coords[len(tenant_a):, 1], c='red', alpha=0.6, label='Tenant B (Healthcare)')
ax.set_title('Multi-Tenant Embedding Space (PCA)')
ax.legend()
plt.tight_layout()
plt.show()

## Isolation Scorecard

In [ ]:
scorecard_data = []
for exp_id, data in experiments.items():
    scorecard_data.append({
        'Experiment': data['experiment_name'],
        'Total Probes': data['total_probes'],
        'Detections': data['cross_tenant_detections'],
        'Detection Rate': data['detection_rate'],
        'Max Score': data['max_similarity_score'],
    })

df = pd.DataFrame(scorecard_data)
df.style.background_gradient(subset=['Detection Rate'], cmap='RdYlGn_r')

## Defense Comparison

In [ ]:
defense_path = results_dir / 'defenses' / 'defense_results.json'
if defense_path.exists():
    with open(defense_path) as f:
        defenses = json.load(f)
    
    df_def = pd.DataFrame(defenses)[['name', 'effectiveness', 'overhead_ms', 'cost_multiplier']]
    df_def.columns = ['Mitigation', 'Effectiveness', 'Overhead (ms)', 'Cost Multiplier']
    display(df_def)
else:
    print('No defense results found. Run: vectorbleed run-defenses')